# Laboratorio 1: Series de Tiempo — AVANCE
**CC3084 – Data Science | UVG | Semestre II 2026**
Jonathan Zacarias 231104
Mario Rocha 23501
**Dataset:** Ingreso de viajeros internacionales a Guatemala (enero 2009 – junio 2026)

Este notebook contiene el **avance** solicitado para el 23 de julio de 2026 (17:20):
1. Análisis exploratorio de datos (EDA) completo.
2. División del conjunto en entrenamiento (70%) y prueba (30%).
3. Construcción de las series de tiempo mensuales (serie obligatoria + dos categorías: **Vías de ingreso** y **Tipo de viajero**).
4. Análisis detallado (incisos a–e del enunciado) de **dos series**: Total mensual y Vía Aérea.

El resto de series, la selección/ajuste de modelos (ARIMA, Prophet, Holt-Winters, suavizamiento exponencial, seasonal naive), las métricas de predicción (MAE, RMSE, AIC, BIC) y el análisis comparativo final se completarán en el documento final del 26 de julio de 2026.

**Nota sobre los datos** (según hoja "Notas" del archivo fuente): el conjunto combina tres tramos con metodologías distintas (2009–2020 respaldos históricos, 2021–2022 entrega del IGM, 2023–jun. 2026 sistema depurado del INGUAT). Existe un **quiebre metodológico entre 2022 y 2023** que reduce artificialmente la categoría "Viajero" (comercio fronterizo/tránsito) sin que represente una caída real de turismo. Para comparar todo el período se recomienda usar **Turista + Excursionista**.


## 0. Configuración e importación de librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
TEAL = "#0B7285"

MESES = {'Ene':1,'Feb':2,'Mar':3,'Abr':4,'May':5,'Jun':6,
         'Jul':7,'Ago':8,'Sep':9,'Oct':10,'Nov':11,'Dic':12}

ModuleNotFoundError: No module named 'pandas'

## 1. Carga de datos

In [ ]:
df = pd.read_excel("Base_Migracion_2009-2026jun.xlsx", sheet_name="Datos")
df["Mes_num"] = df["Mes"].map(MESES)
df["Fecha"] = pd.to_datetime(dict(year=df["Año"], month=df["Mes_num"], day=1))

print("Dimensiones:", df.shape)
df.head()

In [ ]:
df.dtypes

## 2. Análisis Exploratorio de Datos (EDA)

### 2.1 Valores faltantes, duplicados y valores atípicos

In [ ]:
print("Valores faltantes por columna:")
print(df.isna().sum())
print("\nFilas duplicadas exactas:", df.duplicated().sum())
print("Valores negativos en 'Viajero':", (df["Viajero"] < 0).sum())
print("Valores en cero en 'Viajero':", (df["Viajero"] == 0).sum(),
      f"({(df['Viajero']==0).mean():.2%})")

El conjunto **no tiene valores nulos ni filas duplicadas**. Hay 54 registros en cero (0.03%), consistentes con combinaciones categóricas de muy baja frecuencia, no con errores de carga.

In [ ]:
q1, q3 = df["Viajero"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 3 * iqr
outliers = (df["Viajero"] > upper).sum()
print(f"Registros sobre el límite superior IQR (k=3): {outliers} ({outliers/len(df):.2%})")
df["Viajero"].describe()

A nivel de **registro de detalle**, ~13% de las filas quedan marcadas como atípicas por un criterio IQR, pero esto es **esperado y no es un error**: la columna 'Viajero' mezcla flujos muy heterogéneos en la misma columna (p. ej. la frontera '01 La Aurora' o el tipo 'Turista' concentran volúmenes muy superiores a fronteras pequeñas o al tipo 'Cruceristas'). La gran dispersión (media 324.7 vs. mediana 7.0) confirma una distribución muy asimétrica a la derecha. Los atípicos "reales" se evaluarán sobre las **series agregadas mensuales** (nivel de modelado), no sobre el detalle.

### 2.2 Comportamiento temporal del número de viajeros

In [ ]:
serie_total = df.groupby("Fecha")["Viajero"].sum().sort_index()

fig, ax = plt.subplots()
ax.plot(serie_total.index, serie_total.values, color=TEAL)
ax.set_title("Ingreso mensual total de viajeros internacionales a Guatemala (2009-2026)")
ax.set_xlabel("Fecha"); ax.set_ylabel("Viajeros")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x>=1e6 else f"{x/1e3:.0f}k"))
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-12-31"), color="red", alpha=0.08, label="Pandemia COVID-19")
ax.axvline(pd.Timestamp("2023-01-01"), color="orange", linestyle="--", linewidth=1, label="Quiebre metodológico 2023")
ax.legend()
plt.tight_layout(); plt.show()

La serie muestra una **tendencia creciente** entre 2009 y 2019, un **colapso abrupto** en marzo de 2020 por la pandemia (zona roja), un piso prolongado en 2020–2021, y una **recuperación marcada** desde 2022. La línea naranja punteada (enero 2023) marca el quiebre metodológico documentado en las notas del dataset.

In [ ]:
anual = df.groupby("Año")["Viajero"].sum()
fig, ax = plt.subplots()
ax.bar(anual.index.astype(str), anual.values, color=TEAL)
ax.set_title("Total anual de viajeros internacionales (2026 solo ene-jun)")
ax.set_ylabel("Viajeros"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()
anual

In [ ]:
df_comp = df[df["Tipo de Viajero"].isin(["Turista", "Excursionista"])]
serie_comp = df_comp.groupby("Fecha")["Viajero"].sum().sort_index()

fig, ax = plt.subplots()
ax.plot(serie_total.index, serie_total.values, label="Total (todas las categorías)", color="grey", alpha=0.6)
ax.plot(serie_comp.index, serie_comp.values, label="Turista + Excursionista (comparable)", color=TEAL)
ax.set_title("Serie total vs. serie comparable Turista+Excursionista")
ax.legend(); plt.tight_layout(); plt.show()

Al comparar ambas series se confirma que la brecha entre el 'Total' y 'Turista+Excursionista' se **amplía antes de 2023** (por el peso de la categoría 'Viajero' de comercio fronterizo/tránsito) y se **reduce después de 2023**, validando la recomendación de usar la serie comparable para analizar la tendencia de largo plazo en turismo.

**Comportamiento durante y después de la pandemia:** el ingreso total cae de 4.69M de viajeros en 2019 a 1.27M en 2020 (–73%) y se mantiene en un piso similar en 2021. La recuperación es rápida en 2022 (4.32M), aunque en parte impulsada por el cambio metodológico. 2023–2025 se estabiliza entre 3.2M y 3.6M anuales bajo la nueva metodología depurada del INGUAT.

### 2.3 Países con mayor cantidad de viajeros

In [ ]:
top_paises = df.groupby("País")["Viajero"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots()
ax.barh(top_paises.index[::-1], top_paises.values[::-1], color=TEAL)
ax.set_title("Top 10 países/agrupaciones de residencia por viajeros acumulados (2009-2026)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout(); plt.show()
top_paises

El Salvador (16.2M) y Guatemala (14.8M — viajeros identificados con residencia en el propio país, usualmente comercio/tránsito fronterizo) encabezan el acumulado, seguidos de Estados Unidos (7.0M), Honduras (2.8M) y México (1.8M). El dominio de países vecinos centroamericanos es consistente con el peso de la vía terrestre.

### 2.4 Regiones con mayor cantidad de viajeros

In [ ]:
top_regiones = df.groupby("Región dos")["Viajero"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
ax.barh(top_regiones.index[::-1], top_regiones.values[::-1], color=TEAL)
ax.set_title("Viajeros acumulados por región ('Región dos')")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout(); plt.show()
top_regiones

América del Centro concentra 37.4M de viajeros acumulados (~74% del total), seguida por América del Norte (9.4M) y Europa (2.2M). El turismo/tránsito hacia Guatemala es predominantemente **intrarregional**, con Norteamérica como principal mercado de largo alcance.

### 2.5 Vías de ingreso y fronteras más utilizadas

In [ ]:
via = df.groupby("Vía")["Viajero"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
ax.pie(via.values, labels=via.index, autopct="%1.1f%%", colors=["#0B7285","#66A9C9","#B7DCE8"])
ax.set_title("Distribución de viajeros por vía de ingreso")
plt.tight_layout(); plt.show()
via

In [ ]:
top_fronteras = df.groupby("Frontera")["Viajero"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots()
ax.barh(top_fronteras.index[::-1], top_fronteras.values[::-1], color=TEAL)
ax.set_title("Top 10 fronteras por viajeros acumulados")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout(); plt.show()
top_fronteras

La vía **Terrestre** concentra el 61.4% de los viajeros (32.0M), seguida por la **Aérea** (36.6%, 19.1M) y, muy por debajo, la **Marítima** (2.4%, 1.2M) — coherente con la nota del dataset de que la vía Marítima pierde detalle de registro desde 2017.

El aeropuerto **La Aurora** es el punto de ingreso más importante (19.0M), seguido de las fronteras terrestres **Valle Nuevo** con El Salvador (10.7M) y **San Cristóbal** con Honduras (5.4M). Estas tres fronteras concentran cerca del 68% del total acumulado.

### 2.6 Tipo de viajero

In [ ]:
tipo = df.groupby("Tipo de Viajero")["Viajero"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
ax.bar(tipo.index, tipo.values, color=TEAL)
ax.set_title("Viajeros acumulados por tipo de viajero")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
plt.tight_layout(); plt.show()
tipo

In [ ]:
tipo_anual = df.pivot_table(index="Año", columns="Tipo de Viajero", values="Viajero", aggfunc="sum").fillna(0)
fig, ax = plt.subplots()
tipo_anual.plot(ax=ax, marker="o")
ax.set_title("Viajeros anuales por Tipo de Viajero (quiebre metodológico 2022→2023)")
ax.set_ylabel("Viajeros")
plt.tight_layout(); plt.show()

## 3. División en entrenamiento y prueba

Al tratarse de series de tiempo, la división se hace de forma **cronológica** (no aleatoria), evitando fuga de información del futuro al pasado. Con 210 meses disponibles se usa un corte aproximado 70/30.

In [ ]:
fechas = sorted(df["Fecha"].unique())
n_train = int(len(fechas) * 0.7)
fecha_corte = fechas[n_train - 1]

train_df = df[df["Fecha"] <= fecha_corte].copy()
test_df = df[df["Fecha"] > fecha_corte].copy()

print(f"Total de meses: {len(fechas)}")
print(f"Entrenamiento: {len(train_df['Fecha'].unique())} meses "
      f"({fechas[0]:%Y-%m} a {fecha_corte:%Y-%m}) -> {len(train_df['Fecha'].unique())/len(fechas):.1%}")
print(f"Prueba:        {len(test_df['Fecha'].unique())} meses "
      f"({fechas[n_train]:%Y-%m} a {fechas[-1]:%Y-%m}) -> {len(test_df['Fecha'].unique())/len(fechas):.1%}")

El corte cae en plena fase de recuperación pos-pandemia (marzo 2021). Esto implica que el conjunto de prueba abarcará tanto el resto de la recuperación 2021–2022 como el nuevo régimen metodológico 2023–2026 — un reto interesante para los modelos, que se discutirá en el documento final. **Todas las series del punto 4 se construyen únicamente con el conjunto de entrenamiento.**

## 4. Construcción de series de tiempo mensuales

Serie obligatoria (Total mensual) + dos categorías seleccionadas: **(i) Vías de ingreso** y **(ii) Tipo de viajero**.

In [ ]:
def monthly_series(data, filt=None):
    d = data if filt is None else data[filt]
    s = d.groupby("Fecha")["Viajero"].sum().sort_index()
    s = s.asfreq("MS", fill_value=0)   # asegura continuidad mensual
    return s

series = {}
series["Total"] = monthly_series(train_df)

# Categoría 1: Vías de ingreso
for via_name in ["Aérea", "Terrestre", "Marítima"]:
    series[f"Via_{via_name}"] = monthly_series(train_df, train_df["Vía"] == via_name)

# Categoría 2: Tipo de viajero
for tipo_name in train_df["Tipo de Viajero"].unique():
    series[f"Tipo_{tipo_name}"] = monthly_series(train_df, train_df["Tipo de Viajero"] == tipo_name)

resumen = pd.DataFrame([
    {"Serie": k, "Inicio": v.index.min().strftime("%Y-%m"),
     "Fin": v.index.max().strftime("%Y-%m"), "N": len(v), "Frecuencia": "Mensual"}
    for k, v in series.items()
])
resumen

## 5. Análisis detallado de dos series de tiempo

Se presenta el análisis completo (incisos **a–e** del enunciado) de dos series: el **Total mensual de viajeros** y la **Vía Aérea**. El resto de series se analizará con el mismo procedimiento en el documento final, junto con la selección de modelos (inciso f en adelante).

In [ ]:
def analizar_serie(s, nombre):
    print(f"================ {nombre} ================")
    print(f"Inicio: {s.index.min():%Y-%m}  |  Fin: {s.index.max():%Y-%m}  |  "
          f"Frecuencia: Mensual  |  N={len(s)}")

    # b. Grafico
    fig, ax = plt.subplots()
    ax.plot(s.index, s.values, color=TEAL)
    ax.set_title(f"Serie mensual: {nombre}")
    ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-12-31"), color="red", alpha=0.08)
    plt.tight_layout(); plt.show()

    # c. Descomposicion STL
    stl = STL(s, period=12, robust=True).fit()
    fig = stl.plot()
    fig.set_size_inches(11, 8)
    fig.suptitle(f"Descomposición STL: {nombre}")
    plt.tight_layout(); plt.show()

    var_pre = s[:"2019-12"].var()
    var_pand = s["2020-01":].var()
    print(f"\nVarianza pre-pandemia (2009-2019): {var_pre:,.0f}")
    print(f"Varianza pandemia/post (2020+, dentro de train): {var_pand:,.0f}")
    print(f"Razón varianza pandemia/pre: {var_pand/var_pre:.2f}")

    # e. ACF/PACF
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(s, lags=36, ax=axes[0])
    plot_pacf(s, lags=36, ax=axes[1], method="ywm")
    axes[0].set_title(f"ACF - {nombre}"); axes[1].set_title(f"PACF - {nombre}")
    plt.tight_layout(); plt.show()

    # ADF nivel y primera diferencia
    adf_level = adfuller(s, autolag="AIC")
    s_diff = s.diff().dropna()
    adf_diff = adfuller(s_diff, autolag="AIC")

    print(f"\nADF (nivel):          estadístico={adf_level[0]:.3f}, p-valor={adf_level[1]:.4f}, "
          f"lags={adf_level[2]}, crit. 5%={adf_level[4]['5%']:.3f}")
    print(f"ADF (1a diferencia):   estadístico={adf_diff[0]:.3f}, p-valor={adf_diff[1]:.4f}, "
          f"lags={adf_diff[2]}, crit. 5%={adf_diff[4]['5%']:.3f}")

    return {"nombre": nombre, "var_pre": var_pre, "var_pandemia": var_pand,
            "adf_level_p": adf_level[1], "adf_diff_p": adf_diff[1]}

### 5.1 Serie: Total mensual de viajeros

In [ ]:
res_total = analizar_serie(series["Total"], "Total mensual de viajeros")

a. Inicio, fin y frecuencia: La serie va de enero de 2009 a marzo de 2021, con una frecuencia mensual, para un total de 147 observaciones.

b. Gráfico: El comportamiento es muy parecido al de la serie Total. Se observa un crecimiento constante entre 2009 y 2019, una caída muy fuerte durante 2020 y luego una recuperación hacia el final del período de entrenamiento. También se aprecia una estacionalidad anual bastante marcada.

c. Descomposición y estacionariedad: Al descomponer la serie, la tendencia vuelve a mostrar el impacto del 2020, lo que indica que la media no es estacionaria. Además, la relación entre la varianza durante la pandemia y antes de ella es incluso mayor (≈4.0), lo que confirma que el transporte aéreo fue el más afectado por el cierre de fronteras. Esto también sugiere que la varianza no es estacionaria.

d. ¿Transformar la serie? Sí. Al igual que con la serie Total, conviene aplicar una transformación que ayude a estabilizar la varianza y luego realizar una diferenciación regular (d≥1). También vale la pena revisar si hace falta una diferenciación estacional.

e. No estacionariedad en media (ACF y ADF): La ACF disminuye de forma lenta, sin un corte claro, lo que es típico de una serie no estacionaria. Por su parte, la prueba ADF en el nivel original arroja un p-valor de aproximadamente 0.15, por lo que no se puede considerar estacionaria. Después de aplicar una diferencia, el p-valor baja a cerca de 0.001, indicando que la serie ya es estacionaria. En este caso, una diferenciación (d=1) es suficiente.

### 5.2 Serie: Vía Aérea

In [ ]:
res_aerea = analizar_serie(series["Via_Aérea"], "Vía Aérea")

In [ ]:
pd.DataFrame([res_total, res_aerea])